## Minimo V0 e minimo BRT

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

from pathlib import Path
import json
import sys
from time import perf_counter

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

print("Dispositivi JAX:", jax.devices())
assert jax.default_backend() == "cpu"

PROJECT_ROOT = next(
    (
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "hj_reachability" / "systems" / "relative_vehicle_6d.py"
        ).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Aprire il notebook dalla cartella del repository o da tests."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import hj_reachability as hj
from hj_reachability.systems.relative_vehicle_6d import RelativeVehicle6D

BRT_FILE_1 = PROJECT_ROOT / "results" / "brt" / "eggert.npz"
BRT_FILE_2 = PROJECT_ROOT / "results" / "brt" / "ttc.npz"

SAVE_RESULTS = True

# Nomi dei due output, senza estensione .npz
BRT_NAME_PROPAGATED = "eggert_ttc_from_v0"
BRT_NAME_MINIMUM = "eggert_ttc_minimum"

OUTPUT_DIRECTORY = PROJECT_ROOT / "results" / "brt"

STATE_NAMES = (
    "x_rel", "y_rel", "theta_rel", "v_H", "delta_E", "v_E"
)

Dispositivi JAX: [CpuDevice(id=0)]


In [2]:
def load_brt(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"File BRT non trovato: {path}")
    keys = (
        "V0", "BRT", "grid_lo", "grid_hi", "grid_shape", "periodic_dims",
        "initial_time", "target_time", "state_names", "metadata_json", *STATE_NAMES,
    )
    with np.load(path, allow_pickle=False) as data:
        missing = set(keys).difference(data.files)
        if missing:
            raise ValueError(f"{path.name}: chiavi mancanti: {sorted(missing)}")
        result = {key: data[key].copy() for key in keys}
    result["metadata"] = json.loads(str(result.pop("metadata_json").item()))
    result["path"] = str(path.resolve())
    shape = tuple(int(n) for n in result["grid_shape"])
    if len(shape) != 6 or any(n < 2 for n in shape):
        raise ValueError(f"{path.name}: griglia 6D non valida: {shape}")
    if tuple(result["state_names"].tolist()) != STATE_NAMES:
        raise ValueError(f"{path.name}: ordine degli stati diverso da quello atteso.")
    lo, hi = result["grid_lo"], result["grid_hi"]
    if (lo.shape != (6,) or hi.shape != (6,)
            or not np.all(np.isfinite(lo)) or not np.all(np.isfinite(hi))
            or not np.all(hi > lo)):
        raise ValueError(f"{path.name}: dominio non valido.")
    periodic = tuple(result["periodic_dims"].tolist())
    if len(set(periodic)) != len(periodic) or any(d not in range(6) for d in periodic):
        raise ValueError(f"{path.name}: periodic_dims non valido.")
    for key in ("V0", "BRT"):
        values = result[key]
        if values.shape != shape or not np.all(np.isfinite(values)):
            raise ValueError(f"{path.name}: {key} ha forma errata o valori non finiti.")
    for dim, name in enumerate(STATE_NAMES):
        expected = np.linspace(lo[dim], hi[dim], shape[dim], endpoint=dim not in periodic)
        if (result[name].shape != (shape[dim],)
                or not np.allclose(result[name], expected, rtol=1e-6, atol=1e-6)):
            raise ValueError(f"{path.name}: coordinate {name} incoerenti con il dominio.")
    meta = result["metadata"]
    if (meta["dynamics"]["class"] != "RelativeVehicle6D"
            or meta["dynamics"]["module"] != RelativeVehicle6D.__module__):
        raise ValueError(f"{path.name}: dinamica non supportata.")
    solver = meta["solver"]
    if solver["hamiltonian_postprocessor"] != "backwards_reachable_tube":
        raise ValueError(f"{path.name}: il file non usa la propagazione BRT attesa.")
    for name in ("initial_time", "target_time"):
        if not np.isclose(float(result[name]), solver[name], rtol=1e-6, atol=1e-6):
            raise ValueError(f"{path.name}: {name} incoerente con i metadati.")
    if not np.isfinite(solver["initial_time"]) or not np.isfinite(solver["target_time"]):
        raise ValueError(f"{path.name}: tempi non finiti.")
    if solver["target_time"] >= solver["initial_time"]:
        raise ValueError(f"{path.name}: la propagazione deve procedere all'indietro.")
    return result


def check_compatibility(first, second):
    for key in ("state_names", "grid_shape", "periodic_dims"):
        if not np.array_equal(first[key], second[key]):
            raise ValueError(f"Input incompatibili: {key} differisce.")
    for key in ("grid_lo", "grid_hi", "initial_time", "target_time", *STATE_NAMES):
        if not np.allclose(first[key], second[key], rtol=1e-6, atol=1e-6):
            raise ValueError(f"Input incompatibili: {key} differisce.")
    for section in ("dynamics", "solver"):
        if first["metadata"][section] != second["metadata"][section]:
            raise ValueError(f"Input incompatibili: metadati {section} diversi.")


input_1 = load_brt(BRT_FILE_1)
input_2 = load_brt(BRT_FILE_2)
check_compatibility(input_1, input_2)
print("Input compatibili. Griglia:", tuple(input_1["grid_shape"]))

Input compatibili. Griglia: (np.int32(30), np.int32(17), np.int32(48), np.int32(8), np.int32(11), np.int32(8))


In [3]:
def normalize(values):
    maximum_abs = float(np.max(np.abs(values)))
    divisor = maximum_abs if maximum_abs > 0.0 else 1.0
    normalized = np.asarray(values / divisor, dtype=np.float32)
    if not np.all(np.isfinite(normalized)) or np.max(np.abs(normalized)) > 1.0 + 1e-6:
        raise ValueError("Normalizzazione non valida.")
    return normalized, divisor


V01_normalized, scale_V01 = normalize(input_1["V0"])
V02_normalized, scale_V02 = normalize(input_2["V0"])
BRT1_normalized, scale_BRT1 = normalize(input_1["BRT"])
BRT2_normalized, scale_BRT2 = normalize(input_2["BRT"])

V0_min = np.minimum(V01_normalized, V02_normalized)
BRT_min = np.minimum(BRT1_normalized, BRT2_normalized)
normalization_scales = {
    "V01": scale_V01, "V02": scale_V02,
    "BRT1": scale_BRT1, "BRT2": scale_BRT2,
}
for name, values in (
    ("V01_normalized", V01_normalized), ("V02_normalized", V02_normalized),
    ("BRT1_normalized", BRT1_normalized), ("BRT2_normalized", BRT2_normalized),
    ("V0_min", V0_min), ("BRT_min", BRT_min),
):
    print(f"{name:18s}: [{values.min():.6f}, {values.max():.6f}]")
print("Divisori:", normalization_scales)

V01_normalized    : [-0.250000, 1.000000]
V02_normalized    : [-1.000000, 1.000000]
BRT1_normalized   : [-1.000000, 0.084791]
BRT2_normalized   : [-1.000000, 0.234781]
V0_min            : [-1.000000, 1.000000]
BRT_min           : [-1.000000, 0.084791]
Divisori: {'V01': 0.800000011920929, 'V02': 3.0, 'BRT1': 9.434988975524902, 'BRT2': 12.777870178222656}


In [4]:
metadata = input_1["metadata"]
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(lo=jnp.asarray(input_1["grid_lo"]), hi=jnp.asarray(input_1["grid_hi"])),
    shape=tuple(int(n) for n in input_1["grid_shape"]),
    periodic_dims=tuple(int(d) for d in input_1["periodic_dims"]),
)
for name, vector in zip(STATE_NAMES, grid.coordinate_vectors):
    if not np.allclose(np.asarray(vector), input_1[name], rtol=1e-6, atol=1e-6):
        raise ValueError(f"Griglia ricostruita incoerente: {name}.")

dynamics = RelativeVehicle6D(**metadata["dynamics"]["parameters"])
solver_settings = hj.SolverSettings.with_accuracy(
    metadata["solver"]["accuracy"],
    hamiltonian_postprocessor=hj.solver.backwards_reachable_tube,
)
initial_time = float(input_1["initial_time"])
target_time = float(input_1["target_time"])

print(f"Inizio propagazione: t={initial_time:g} → {target_time:g} s", flush=True)
start = perf_counter()
BRT_from_min_V0 = np.asarray(hj.step(
    solver_settings, dynamics, grid, initial_time,
    jnp.asarray(V0_min), target_time, progress_bar=False,
), dtype=np.float32)
if BRT_from_min_V0.shape != V0_min.shape or not np.all(np.isfinite(BRT_from_min_V0)):
    raise ValueError("Il BRT propagato ha forma errata o valori non finiti.")
print(f"Fine propagazione: {perf_counter() - start:.2f} s")
print(f"BRT_from_min_V0: [{BRT_from_min_V0.min():.6f}, {BRT_from_min_V0.max():.6f}]")

Inizio propagazione: t=0 → -3 s
Fine propagazione: 1371.96 s
BRT_from_min_V0: [-12.770407, 1.000000]


In [5]:
from datetime import datetime, timezone

if SAVE_RESULTS:
    output_names = (
        BRT_NAME_PROPAGATED,
        BRT_NAME_MINIMUM,
    )

    for name in output_names:
        if (
            not isinstance(name, str)
            or not name.strip()
            or name in {".", ".."}
            or "/" in name
            or "\\" in name
            or name.lower().endswith(".npz")
        ):
            raise ValueError(
                "Inserire nomi validi, senza cartelle o estensione .npz."
            )

    output_paths = [
        OUTPUT_DIRECTORY / f"{name}.npz"
        for name in output_names
    ]

    if output_paths[0].resolve() == output_paths[1].resolve():
        raise ValueError("I due output devono avere nomi diversi.")

    input_paths = {
        BRT_FILE_1.resolve(),
        BRT_FILE_2.resolve(),
    }

    for path in output_paths:
        if path.resolve() in input_paths:
            raise ValueError(f"L'output coincide con un input: {path}")

        if path.exists():
            raise FileExistsError(
                f"Il file esiste già: {path}. Scegliere un altro nome."
            )

    common_arrays = {
        key: input_1[key]
        for key in (
            "grid_lo",
            "grid_hi",
            "grid_shape",
            "periodic_dims",
            "initial_time",
            "target_time",
            "state_names",
            *STATE_NAMES,
        )
    }

    outputs = (
        (
            output_paths[0],
            BRT_NAME_PROPAGATED,
            BRT_from_min_V0,
            "HJ propagation of minimum(V01_normalized, V02_normalized)",
        ),
        (
            output_paths[1],
            BRT_NAME_MINIMUM,
            BRT_min,
            "minimum(BRT1_normalized, BRT2_normalized)",
        ),
    )

    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    for path, name, values, construction in outputs:
        output_metadata = json.loads(json.dumps(input_1["metadata"]))

        output_metadata["created_at"] = (
            datetime.now(timezone.utc).isoformat()
        )

        output_metadata["metric"] = {
            "name": name,
            "parameters": {},
            "terminal_value_definition": (
                "minimum(V01 / max_abs(V01), V02 / max_abs(V02))"
            ),
        }

        output_metadata["combination"] = {
            "sources": [input_1["path"], input_2["path"]],
            "BRT_construction": construction,
            "propagated_from_saved_V0": path == output_paths[0],
        }

        output_metadata["normalization"] = {
            "method": "independent_max_abs",
            "divisors": normalization_scales,
            "output_BRT_renormalized": False,
        }

        output_metadata["arrays"] = {
            "V0_shape": list(V0_min.shape),
            "BRT_shape": list(values.shape),
            "storage_dtype": "float32",
            "gradients_available": False,
        }

        with path.open("xb") as output:
            np.savez_compressed(
                output,
                V0=np.asarray(V0_min, dtype=np.float32),
                BRT=np.asarray(values, dtype=np.float32),
                metric_name=np.asarray(name),
                metadata_json=np.asarray(
                    json.dumps(output_metadata, indent=2)
                ),
                **common_arrays,
            )

        print(f"BRT salvato: {path}")

else:
    print("Salvataggio disattivato: risultati disponibili in memoria.")

BRT salvato: /home/alessio-grisorio/TESI/hj_reachability/results/brt/eggert_ttc_from_v0.npz
BRT salvato: /home/alessio-grisorio/TESI/hj_reachability/results/brt/eggert_ttc_minimum.npz
